# Session1_Task7 — Customer Segmentation & Recommendation

## คำสั่งสำคัญที่ใช้
- `KMeans(n_clusters=3)` → แบ่งลูกค้าออกเป็น 3 กลุ่มตามพฤติกรรมการซื้อ
- `StandardScaler()` → ปรับสเกลข้อมูลให้มีค่าเฉลี่ย=0 ก่อน clustering
- `itertools.combinations()` → หาคู่สินค้าที่ซื้อพร้อมกัน
- `Counter()` → นับความถี่ของแต่ละคู่สินค้า

In [1]:
import pandas as pd
import numpy as np
import warnings; warnings.filterwarnings('ignore')
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from itertools import combinations
from collections import Counter

In [2]:
# โหลดข้อมูล
s = pd.read_csv('sales_transactions_cleaned.csv')
c = pd.read_csv('customers_cleaned.csv')
s['revenue'] = (s['quantity'] * s['price']) - pd.to_numeric(s['discount_amount'], errors='coerce').fillna(0)

In [3]:
# --- 1. Feature Engineering ---

# นับจำนวน transaction และค่าเฉลี่ยต่อ transaction ของแต่ละลูกค้า
feat = s.groupby('customer_id').agg(
    total_purchases   =('transaction_id', 'nunique'),  # นับ transaction ไม่ซ้ำ
    avg_purchase_value=('revenue',         'mean')     # ค่าเฉลี่ย revenue ต่อ transaction
).reset_index()

display(feat.head())

,customer_id,total_purchases,avg_purchase_value
0,101,13,8.484615
1,102,17,19.315294
2,103,22,24.063182
3,104,21,8.260000
4,105,11,13.642727


In [4]:
# --- 2. K-Means Clustering ---

# StandardScaler → ปรับสเกล ทำให้ทุกฟีเจอร์มีค่าเฉลี่ย 0 และ std 1
# จำเป็นเพราะ KMeans ใช้ระยะทาง ถ้า scale ต่างกันมากจะ bias
scaler = StandardScaler()
X = scaler.fit_transform(feat[['total_purchases', 'avg_purchase_value']])
# .fit_transform() → เรียนรู้ค่า mean/std แล้วแปลงข้อมูลพร้อมกัน

# KMeans(n_clusters=3) → แบ่งออกเป็น 3 กลุ่ม
# random_state=42 → ตั้ง seed ให้ผลลัพธ์เหมือนกันทุกครั้ง
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
feat['cluster_label'] = kmeans.fit_predict(X) + 1  # +1 ให้ label เริ่มที่ 1 แทน 0
# .fit_predict() → ฝึก model และทำนาย cluster ของข้อมูลพร้อมกัน

print('Cluster distribution:')
print(feat['cluster_label'].value_counts().sort_index())

Cluster distribution:
cluster_label
1    255
2    286
3     79
Name: count, dtype: int64


In [5]:
# --- 3. Product Affinity (สินค้าที่ซื้อพร้อมกัน) ---

# หา top 3 สินค้าที่ซื้อพร้อมกันในแต่ละ transaction
# groupby transaction → รวม product_id ที่อยู่ใน transaction เดียวกัน
tx_products = s.groupby('transaction_id')['product_id'].apply(list)

# นับคู่สินค้าที่ปรากฏพร้อมกัน
pair_counts = Counter()
for products in tx_products:
    if len(products) >= 2:
        # combinations(list, 2) → หาทุกคู่ที่เป็นไปได้จาก list
        for pair in combinations(sorted(products), 2):
            pair_counts[pair] += 1

# สร้าง dict: product_id → top 3 สินค้าที่ซื้อพร้อมกันบ่อยที่สุด
all_products = s['product_id'].dropna().unique()
affinity = {}
for pid in all_products:
    related = {}
    for (a, b), cnt in pair_counts.items():
        if a == pid: related[b] = cnt
        if b == pid: related[a] = cnt
    # เรียงจากมากไปน้อย เลือก top 3
    affinity[pid] = sorted(related, key=related.get, reverse=True)[:3]

print('Affinity sample (product 1):', affinity.get(1, []))

Affinity sample (product 1): []


In [6]:
# --- 4. Recommendation per customer ---

# merge customer กับ cluster
cust_cluster = feat[['customer_id','cluster_label']]

# สินค้าที่ลูกค้าแต่ละคนซื้อแล้ว
purchased = s.groupby('customer_id')['product_id'].apply(set).to_dict()

# หาสินค้าขายดีในแต่ละ cluster
s_cluster = s.merge(cust_cluster, on='customer_id')
cluster_top = (s_cluster.groupby(['cluster_label','product_id'])['quantity']
               .sum().reset_index()
               .sort_values(['cluster_label','quantity'], ascending=[True,False]))

rows = []
for _, row in cust_cluster.iterrows():
    cid, cl = row['customer_id'], row['cluster_label']
    bought  = purchased.get(cid, set())
    # แนะนำสินค้าที่ยังไม่เคยซื้อ และขายดีในกลุ่มเดียวกัน
    recs = [int(p) for p in cluster_top[cluster_top['cluster_label']==cl]['product_id']
            if p not in bought][:3]
    while len(recs) < 3: recs.append(None)  # เติม None ถ้าไม่ครบ 3
    rows.append([int(cid), cl, recs[0], recs[1], recs[2]])

result = pd.DataFrame(rows, columns=['customer_id','cluster_label',
                                      'recommended_product_1','recommended_product_2','recommended_product_3'])
display(result.head())
result.to_csv('Session5_Segmentation_and_Recommendations.csv', index=False)
print('✅ Saved Session5_Segmentation_and_Recommendations.csv')

# === จุดสังเกต ===
# ✔ ไฟล์มี 5 คอลัมน์ตามโจทย์
# ✔ cluster_label มีแค่ 1, 2, 3
# ✔ recommended_product ไม่ควรเป็น product ที่ลูกค้าซื้อแล้ว

,customer_id,cluster_label,recommended_product_1,recommended_product_2,recommended_product_3
0,101,1,6.0,3.0,2.0
1,102,2,9.0,18.0,20.0
2,103,2,9.0,18.0,20.0
3,104,2,9.0,18.0,20.0
4,105,1,3.0,2.0,9.0


✅ Saved Session5_Segmentation_and_Recommendations.csv
